# Extract meteorological data at the Longmen Hydrological Station

In [ ]:

import netCDF4 as nc
import pandas as pd
import numpy as np

lon_target = 110.35   
lat_target = 35.40  

file_paths = [
    "cru_ts4.09.1901.2024.cld.dat.nc",
    "cru_ts4.09.1901.2024.dtr.dat.nc",
    "cru_ts4.09.1901.2024.frs.dat.nc",
    "cru_ts4.09.1901.2024.pet.dat.nc",
    "cru_ts4.09.1901.2024.pre.dat.nc",
    "cru_ts4.09.1901.2024.tmn.dat.nc",
    "cru_ts4.09.1901.2024.tmp.dat.nc",
    "cru_ts4.09.1901.2024.tmx.dat.nc",
    "cru_ts4.09.1901.2024.vap.dat.nc",
    "cru_ts4.09.1901.2024.wet.dat.nc"
]

all_data = []

for file_path in file_paths:

    dataset = nc.Dataset(file_path)
    

    lons = dataset.variables['lon'][:]
    lats = dataset.variables['lat'][:]
    time = dataset.variables['time'][:]
    
    
    lon_idx = np.argmin(np.abs(lons - lon_target))
    lat_idx = np.argmin(np.abs(lats - lat_target))
    
  
    var_name = file_path.split('.')[4]  
    var_data = dataset.variables[var_name][:, lat_idx, lon_idx]
    

    time_units = dataset.variables['time'].units
    time_cal = dataset.variables['time'].calendar
    dates = nc.num2date(time, units=time_units, calendar=time_cal)
    df = pd.DataFrame({
        'time': dates,
        var_name: var_data
    })
    
    all_data.append(df)
merged_df = pd.concat(all_data, axis=1)
merged_df = merged_df.loc[:, ~merged_df.columns.duplicated()]

# Handle Missing Values

print("\n=== 2. 缺失值检查 ===")
# Check for missing values
missing = merged_df.isnull().sum()
total = len(merged_df)

if missing.sum() == 0:
    print("no missing values")
else:
    print(f"there are {missing.sum()} missing values")
    print("missing values in each column:")
    for column, count in missing.items():
        if count > 0:
            print(f"- {column}: {count} ({count/total*100:.2f}%)")

merged_df.to_csv('longmen_meteorological_data.csv', index=False)

no missing values